In [ ]:
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, f1_score, accuracy_score, recall_score, precision_score
)

from imblearn.over_sampling import SMOTE

RANDOM_STATE = 42


# ─────────────────────────────────────────────────────────
# PAGE CONFIG
# ─────────────────────────────────────────────────────────

st.set_page_config(
    page_title="Telecom Churn Predictor",
    page_icon="📡",
    layout="wide"
)

# ─────────────────────────────────────────────────────────
# MODEL TRAINING
# ─────────────────────────────────────────────────────────

@st.cache_resource
def train_model():

    np.random.seed(RANDOM_STATE)
    n = 7043

    gender = np.random.choice([0, 1], n)
    senior = np.random.choice([0, 1], n, p=[0.84, 0.16])
    partner = np.random.choice([0, 1], n)
    dependents = np.random.choice([0, 1], n)
    tenure = np.random.randint(0, 72, n)

    phone_svc = np.random.choice([0, 1], n)
    multi_lines = np.random.choice(['No', 'Yes'], n)
    internet = np.random.choice(['DSL', 'Fiber optic', 'No'], n)

    online_sec = np.random.choice(['No', 'Yes'], n)
    online_bkp = np.random.choice(['No', 'Yes'], n)
    dev_prot = np.random.choice(['No', 'Yes'], n)
    tech_sup = np.random.choice(['No', 'Yes'], n)

    stream_tv = np.random.choice(['No', 'Yes'], n)
    stream_mov = np.random.choice(['No', 'Yes'], n)

    contract = np.random.choice(
        ['Month-to-month', 'One year', 'Two year'], n)

    paperless = np.random.choice([0, 1], n)

    payment = np.random.choice([
        'Electronic check',
        'Mailed check',
        'Bank transfer (automatic)',
        'Credit card (automatic)'
    ], n)

    monthly_chg = np.random.uniform(18, 119, n)
    total_chg = monthly_chg * tenure

    churn_prob = (
        0.05 +
        0.25 * (contract == 'Month-to-month') +
        0.10 * (internet == 'Fiber optic') +
        0.08 * (payment == 'Electronic check') +
        0.08 * senior -
        0.10 * (tenure > 36)
    )

    churn_prob = np.clip(churn_prob, 0, 0.9)

    churn = (np.random.rand(n) < churn_prob).astype(int)

    df = pd.DataFrame({

        'gender': gender,
        'SeniorCitizen': senior,
        'Partner': partner,
        'Dependents': dependents,
        'tenure': tenure,

        'PhoneService': phone_svc,
        'MultipleLines': multi_lines,
        'InternetService': internet,

        'OnlineSecurity': online_sec,
        'OnlineBackup': online_bkp,
        'DeviceProtection': dev_prot,
        'TechSupport': tech_sup,

        'StreamingTV': stream_tv,
        'StreamingMovies': stream_mov,

        'Contract': contract,
        'PaperlessBilling': paperless,
        'PaymentMethod': payment,

        'MonthlyCharges': monthly_chg,
        'TotalCharges': total_chg,
        'Churn': churn
    })

    X = df.drop("Churn", axis=1)
    y = df["Churn"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        stratify=y,
        random_state=RANDOM_STATE
    )

    train_median_charge = X_train["MonthlyCharges"].median()

    def engineer_features(df, median_charge):

        df = df.copy()

        svc_cols = [
            "MultipleLines", "OnlineSecurity",
            "OnlineBackup", "DeviceProtection",
            "TechSupport", "StreamingTV",
            "StreamingMovies"
        ]

        for col in svc_cols:
            if col in df.columns:
                df[col] = df[col].replace({
                    "No phone service": "No",
                    "No internet service": "No"
                })

        df["IsFirstYear"] = (df["tenure"] <= 12).astype(int)

        df["AvgMonthlyCharge"] = df.apply(
            lambda x:
            x["TotalCharges"] / x["tenure"]
            if x["tenure"] > 0 else x["MonthlyCharges"],
            axis=1
        )

        df["FiberOpticUser"] = (df["InternetService"] ==
                                "Fiber optic").astype(int)

        df["IsMonthToMonth"] = (df["Contract"] ==
                                "Month-to-month").astype(int)

        df["HighCostLowTenure"] = (
            (df["MonthlyCharges"] > median_charge)
            & (df["tenure"] < 12)
        ).astype(int)

        return df

    X_train = engineer_features(X_train, train_median_charge)
    X_test = engineer_features(X_test, train_median_charge)

    X_train = pd.get_dummies(X_train)
    X_test = pd.get_dummies(X_test)

    X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

    feature_columns = X_train.columns.tolist()

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    smote = SMOTE(random_state=RANDOM_STATE)

    X_res, y_res = smote.fit_resample(X_train_scaled, y_train)

    model = LogisticRegression(
        penalty="l1",
        solver="liblinear",
        max_iter=1000
    )

    model.fit(X_res, y_res)

    y_prob = model.predict_proba(X_test_scaled)[:, 1]

    y_pred = (y_prob >= 0.5).astype(int)

    fpr, tpr, _ = roc_curve(y_test, y_prob)

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_prob),
        "cm": confusion_matrix(y_test, y_pred),
        "fpr": fpr,
        "tpr": tpr
    }

    return model, scaler, feature_columns, train_median_charge, engineer_features, metrics


# ─────────────────────────────────────────────────────────
# PREDICTION FUNCTION
# ─────────────────────────────────────────────────────────

def predict_churn(data, model, scaler, feature_columns,
                  median_charge, engineer_features):

    df = pd.DataFrame([data])

    df = engineer_features(df, median_charge)

    df = pd.get_dummies(df)

    for col in feature_columns:
        if col not in df.columns:
            df[col] = 0

    df = df[feature_columns]

    X_scaled = scaler.transform(df)

    prob = model.predict_proba(X_scaled)[0][1]

    pred = int(prob >= 0.5)

    return pred, prob


# ─────────────────────────────────────────────────────────
# MAIN APP
# ─────────────────────────────────────────────────────────

def main():

    st.title("📡 Telecom Customer Churn Predictor")

    model, scaler, feature_columns, median_charge, engineer_features, metrics = train_model()

    st.sidebar.title("Navigation")

    page = st.sidebar.radio(
        "",
        ["Predict Churn", "Model Performance"]
    )

    if page == "Predict Churn":

        st.header("Customer Details")

        col1, col2 = st.columns(2)

        with col1:

            gender = st.selectbox("Gender", ["Male", "Female"])
            senior = st.selectbox("Senior Citizen", ["No", "Yes"])
            tenure = st.slider("Tenure", 0, 72, 12)

            internet = st.selectbox(
                "Internet Service",
                ["DSL", "Fiber optic", "No"]
            )

        with col2:

            contract = st.selectbox(
                "Contract",
                ["Month-to-month", "One year", "Two year"]
            )

            payment = st.selectbox(
                "Payment Method",
                [
                    "Electronic check",
                    "Mailed check",
                    "Bank transfer (automatic)",
                    "Credit card (automatic)"
                ]
            )

            monthly = st.number_input(
                "Monthly Charges",
                0.0,
                200.0,
                70.0
            )

        if st.button("Predict"):

            data = {

                "gender": 1 if gender == "Female" else 0,
                "SeniorCitizen": 1 if senior == "Yes" else 0,
                "Partner": 0,
                "Dependents": 0,
                "tenure": tenure,

                "PhoneService": 1,
                "MultipleLines": "No",

                "InternetService": internet,

                "OnlineSecurity": "No",
                "OnlineBackup": "No",
                "DeviceProtection": "No",
                "TechSupport": "No",

                "StreamingTV": "No",
                "StreamingMovies": "No",

                "Contract": contract,
                "PaperlessBilling": 1,
                "PaymentMethod": payment,

                "MonthlyCharges": monthly,
                "TotalCharges": monthly * tenure
            }

            pred, prob = predict_churn(
                data,
                model,
                scaler,
                feature_columns,
                median_charge,
                engineer_features
            )

            if pred == 1:

                st.error(
                    f"⚠ High churn risk\nProbability: {prob:.2%}"
                )

            else:

                st.success(
                    f"✅ Low churn risk\nProbability: {prob:.2%}"
                )

    else:

        st.header("Model Performance")

        st.write("Accuracy:", metrics["accuracy"])
        st.write("Recall:", metrics["recall"])
        st.write("Precision:", metrics["precision"])
        st.write("F1:", metrics["f1"])
        st.write("ROC-AUC:", metrics["roc_auc"])

        fig, ax = plt.subplots()

        sns.heatmap(
            metrics["cm"],
            annot=True,
            fmt="d",
            cmap="Blues",
            ax=ax
        )

        st.pyplot(fig)


if __name__ == "__main__":
    main()

2026-03-13 12:14:06.085 
  command:

    streamlit run C:\Users\PRANATHI SRI\anaconda3\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
  File "C:\Users\PRANATHI SRI\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\PRANATHI SRI\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\PRANATHI SRI\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\PRANATHI SRI\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
2026-03-13 12:14:07.769 `label` got an empty value. This is discouraged for accessibility reasons and may be disallowed in the future by 